# PBMC training example

This notebook demonstrates the complete PBMC workflow using the shared settings in `settinng.py` and the unified training entry point in `train.py`.

In [ ]:
from pathlib import Path
import sys
import importlib.util

start = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in [start, *start.parents] if (path / "settinng.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the scHetGTL repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_modules = [
    "numpy",
    "pandas",
    "scipy",
    "sklearn",
    "scanpy",
    "torch",
    "torch_geometric",
]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ImportError(
        "Missing required Python modules: "
        + ", ".join(missing_modules)
        + ". Use the scHetGTL environment created from environment.yml."
    )

print(f"Project root: {PROJECT_ROOT}")

## Load the fixed PBMC configuration

Only the dataset is selected here. All model and optimization parameters are shared across datasets.

In [ ]:
import pandas as pd

from settinng import get_dataset_config

config = get_dataset_config("pbmc")

path_keys = [
    "rna_path",
    "atac_path",
    "atac_neighbor_path",
    "anchor_path",
    "output_dir",
]

pd.DataFrame(
    {"value": [config[key] for key in path_keys]},
    index=path_keys,
)

In [ ]:
training_keys = [
    "seed",
    "epochs",
    "lr",
    "batch_size",
    "knn_k",
    "n_top_genes",
    "hidden_dim",
    "out_dim",
    "num_layers",
    "num_neighbors",
    "w_geo",
    "w_cls",
    "w_recon",
    "w_center",
    "w_cl_intra",
    "w_cl_inter",
    "tau",
]

pd.DataFrame(
    {"value": [config[key] for key in training_keys]},
    index=training_keys,
)

## Check the downloaded data

In [ ]:
required_keys = [
    "rna_path",
    "atac_path",
    "atac_neighbor_path",
    "anchor_path",
]

file_status = pd.DataFrame(
    [
        {
            "input": key,
            "path": config[key],
            "exists": Path(config[key]).is_file(),
            "size_mb": round(Path(config[key]).stat().st_size / 1024**2, 2)
            if Path(config[key]).is_file()
            else None,
        }
        for key in required_keys
    ]
)

display(file_status)

missing = file_status.loc[~file_status["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(
        "Missing PBMC files. Download and extract the dataset under data/data_pbmc/."
    )

## Inspect the PBMC inputs

The files are opened in backed mode so this summary does not load the full expression matrices into memory.

In [ ]:
import scanpy as sc

rna = sc.read_h5ad(config["rna_path"], backed="r")
atac = sc.read_h5ad(config["atac_path"], backed="r")
atac_lsi = sc.read_h5ad(config["atac_neighbor_path"], backed="r")

summary = pd.DataFrame(
    [
        {"modality": "RNA", "cells": rna.n_obs, "features": rna.n_vars},
        {"modality": "ATAC gene activity", "cells": atac.n_obs, "features": atac.n_vars},
        {"modality": "ATAC LSI", "cells": atac_lsi.n_obs, "features": atac_lsi.n_vars},
    ]
)
display(summary)

cell_types = pd.DataFrame(
    {
        "RNA cells": rna.obs["cell_type"].astype(str).value_counts(),
        "ATAC cells": atac.obs["cell_type"].astype(str).value_counts(),
    }
 ).fillna(0).astype(int)
display(cell_types)

rna.file.close()
atac.file.close()
atac_lsi.file.close()

## Train scHetGTL

First verify the PyTorch build and selected device. The provided Conda environment uses PyTorch 2.5.1 with CUDA 12.1. Training outputs are written to `results/pbmc/`.

In [ ]:
import torch
from torch_geometric.typing import WITH_PYG_LIB, WITH_TORCH_SPARSE

expected_torch_version = "2.5.1"
expected_cuda_runtime = "12.1"
torch_version = torch.__version__.split("+", 1)[0]

print("PyTorch version:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("PyG sampling backend:", "pyg-lib" if WITH_PYG_LIB else "torch-sparse" if WITH_TORCH_SPARSE else "missing")

if torch_version != expected_torch_version or torch.version.cuda != expected_cuda_runtime:
    raise RuntimeError(
        f"Expected torch=={expected_torch_version}+cu121 with CUDA {expected_cuda_runtime}, "
        f"but found torch=={torch.__version__} with CUDA {torch.version.cuda}."
    )

if not (WITH_PYG_LIB or WITH_TORCH_SPARSE):
    raise ImportError("NeighborLoader requires pyg-lib or torch-sparse in this environment.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Training device: cuda")
else:
    print("Training device: cpu")

## Train and load the learned embeddings

In [ ]:
import subprocess
import numpy as np

command = [sys.executable, "train.py", "--dataset", "pbmc"]
print("Running:", " ".join(command))

process = subprocess.Popen(
    command,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)

output_dir = Path(config["output_dir"])
rna_path = output_dir / "rna_embeddings.npy"
atac_path = output_dir / "atac_embeddings.npy"
missing_outputs = [str(path) for path in [rna_path, atac_path] if not path.is_file()]
if missing_outputs:
    raise FileNotFoundError(
        "Training did not produce the expected embedding files: "
        + ", ".join(missing_outputs)
    )

rna_embeddings = np.load(rna_path)
atac_embeddings = np.load(atac_path)

print("RNA embedding shape:", rna_embeddings.shape)
print("ATAC embedding shape:", atac_embeddings.shape)